In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
from torch.utils.data import DataLoader
import torch.utils.data as data
import torch.nn as nn
import sys
import warnings
from itertools import product

!rm -r /kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git clone https://github.com/KirillVishnyakov/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git -C /kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection log --oneline -1

Cloning into 'Architectural-Biases-in-Time-Series-Anomaly-Detection'...
remote: Enumerating objects: 390, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 390 (delta 12), reused 25 (delta 8), pack-reused 356 (from 1)
Receiving objects: 100% (390/390), 100.54 MiB | 41.02 MiB/s, done.
Resolving deltas: 100% (198/198), done.
07be4af (HEAD -> main, origin/main, origin/HEAD) tweaking models, new thresholding mechanism, new weights and notebooks


In [5]:
sys.path.append("/kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection")
import utils.config as config
config.init("/kaggle/input/datasets/kirillvishnyakov/cats-dataset/data.csv", "/kaggle/Architectural-Biases-in-Time-Series-Anomaly-Detection/checkpoint_dir")

from models.lstm_ae import lstm_ae
import dataset
import train
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)
warnings.filterwarnings("ignore")

2.9.0+cu126


In [3]:
seed = 432
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [15]:
def tune_over_grid(hyperparam_grid, train_dataset, val_dataset, epochs = 10):
    results = []
    
    for lookback_window, embedding_dim_ratio, learning_rate in product(
        hyperparam_grid["lookback_window"],
        hyperparam_grid["embedding_dim_ratio"],
        hyperparam_grid["learning_rate"]
    ):
    
        model = lstm_ae(17, lookback_window, embedding_dim_ratio).to(device)
    
        name = f"lb_{lookback_window}_edr_{embedding_dim_ratio}_lr_{learning_rate}"
    
        weights, val_loss, train_loss = train.fit_forecaster(
            device, model, name, train_dataset, val_dataset, learning_rate, 512, epochs
        )
    
        results.append({
            "val_loss": val_loss,
            "name": name,
            "weights": weights
        })
    results.sort(key=lambda x: x['val_loss'])
    results_best = results[0]
    return results

In [ ]:
train_dataset = dataset.AE_Dataset(device, 256, start=0, end=800000)
val_dataset = dataset.AE_Dataset(
    device, 256, scaler=train_dataset.scaler,
    start=800000, end=1000000, train=False
)

## Tune over latent space ratios

In [16]:
hyperparam_grid = \
{ "lookback_window": [256], 
  "embedding_dim_ratio": [2.0, 2.5, 3.0], 
  "learning_rate": [0.002]
}

results = tune_over_grid(hyperparam_grid, train_dataset, val_dataset)
results.sort(key=lambda x: x['val_loss'])
results_best = results[0]
print(f"best model: {results_best['name']}, {results_best['val_loss']}")

LR Scheduler: 781 warmup steps, 15620 total steps
|lb_256_edr_2.0_lr_0.002| train = 0.1355 | test= 0.1242 | LR: 1.99e-03
|lb_256_edr_2.0_lr_0.002| train = 0.1187 | test= 0.1107 | LR: 1.88e-03
|lb_256_edr_2.0_lr_0.002| train = 0.1092 | test= 0.1066 | LR: 1.68e-03
|lb_256_edr_2.0_lr_0.002| train = 0.1053 | test= 0.1046 | LR: 1.41e-03
|lb_256_edr_2.0_lr_0.002| train = 0.1025 | test= 0.1024 | LR: 1.09e-03
|lb_256_edr_2.0_lr_0.002| train = 0.1006 | test= 0.1010 | LR: 7.67e-04
|lb_256_edr_2.0_lr_0.002| train = 0.0990 | test= 0.0980 | LR: 4.69e-04
|lb_256_edr_2.0_lr_0.002| train = 0.0976 | test= 0.0977 | LR: 2.29e-04
|lb_256_edr_2.0_lr_0.002| train = 0.0969 | test= 0.0972 | LR: 7.36e-05
|lb_256_edr_2.0_lr_0.002| train = 0.0966 | test= 0.0970 | LR: 2.00e-05
LR Scheduler: 781 warmup steps, 15620 total steps
|lb_256_edr_2.5_lr_0.002| train = 0.1389 | test= 0.1248 | LR: 1.99e-03
|lb_256_edr_2.5_lr_0.002| train = 0.1167 | test= 0.1105 | LR: 1.88e-03
|lb_256_edr_2.5_lr_0.002| train = 0.1065 | test=

## Test lr - higher ones since traning is stable and loss is still decreasing

In [24]:
hyperparam_grid = \
{ "lookback_window": [256], 
  "embedding_dim_ratio": [2.5], 
  "learning_rate": [0.003, 0.004, 0.005]
}

results = tune_over_grid(hyperparam_grid, train_dataset, val_dataset, epochs = 15)
results.sort(key=lambda x: x['val_loss'])
results_best = results[0]
print(f"best model: {results_best['name']}, {results_best['val_loss']}")

LR Scheduler: 781 warmup steps, 23430 total steps
|lb_256_edr_2.5_lr_0.003| train = 0.1302 | test= 0.1152 | LR: 2.99e-03
|lb_256_edr_2.5_lr_0.003| train = 0.1085 | test= 0.1002 | LR: 2.92e-03
|lb_256_edr_2.5_lr_0.003| train = 0.0977 | test= 0.0940 | LR: 2.79e-03
|lb_256_edr_2.5_lr_0.003| train = 0.0946 | test= 0.0895 | LR: 2.59e-03
|lb_256_edr_2.5_lr_0.003| train = 0.0921 | test= 0.0887 | LR: 2.35e-03
|lb_256_edr_2.5_lr_0.003| train = 0.0902 | test= 0.0866 | LR: 2.06e-03
|lb_256_edr_2.5_lr_0.003| train = 0.0885 | test= 0.0862 | LR: 1.76e-03
|lb_256_edr_2.5_lr_0.003| train = 0.0878 | test= 0.0860 | LR: 1.43e-03
|lb_256_edr_2.5_lr_0.003| train = 0.0855 | test= 0.0835 | LR: 1.12e-03
|lb_256_edr_2.5_lr_0.003| train = 0.0842 | test= 0.0817 | LR: 8.19e-04
|lb_256_edr_2.5_lr_0.003| train = 0.0835 | test= 0.0811 | LR: 5.54e-04
|lb_256_edr_2.5_lr_0.003| train = 0.0823 | test= 0.0804 | LR: 3.33e-04
|lb_256_edr_2.5_lr_0.003| train = 0.0814 | test= 0.0803 | LR: 1.67e-04
|lb_256_edr_2.5_lr_0.003| t

## Train final model with more epochs - slightly lower lr to account for the cosine decay landscape keeping a higher lr for longer because this time the model is training for 30 epochs

In [25]:
hyperparam_grid = \
{ "lookback_window": [256], 
  "embedding_dim_ratio": [2.5], 
  "learning_rate": [0.0036]
}

results = tune_over_grid(hyperparam_grid, train_dataset, val_dataset, epochs = 30)
results.sort(key=lambda x: x['val_loss'])
results_best = results[0]
print(f"best model: {results_best['name']}, {results_best['val_loss']}")

LR Scheduler: 781 warmup steps, 46860 total steps
|lb_256_edr_2.5_lr_0.0036| train = 0.1336 | test= 0.1227 | LR: 3.60e-03
|lb_256_edr_2.5_lr_0.0036| train = 0.1147 | test= 0.1057 | LR: 3.58e-03
|lb_256_edr_2.5_lr_0.0036| train = 0.1048 | test= 0.0987 | LR: 3.54e-03
|lb_256_edr_2.5_lr_0.0036| train = 0.1002 | test= 0.0987 | LR: 3.48e-03
|lb_256_edr_2.5_lr_0.0036| train = 0.0982 | test= 0.0935 | LR: 3.40e-03
|lb_256_edr_2.5_lr_0.0036| train = 0.0944 | test= 0.0942 | LR: 3.30e-03
|lb_256_edr_2.5_lr_0.0036| train = 0.0928 | test= 0.0875 | LR: 3.19e-03
|lb_256_edr_2.5_lr_0.0036| train = 0.0907 | test= 0.0894 | LR: 3.06e-03
|lb_256_edr_2.5_lr_0.0036| train = 0.0892 | test= 0.0861 | LR: 2.92e-03
|lb_256_edr_2.5_lr_0.0036| train = 0.0889 | test= 0.0864 | LR: 2.76e-03
|lb_256_edr_2.5_lr_0.0036| train = 0.0880 | test= 0.0841 | LR: 2.60e-03
|lb_256_edr_2.5_lr_0.0036| train = 0.0859 | test= 0.0822 | LR: 2.42e-03
|lb_256_edr_2.5_lr_0.0036| train = 0.0851 | test= 0.0819 | LR: 2.24e-03
|lb_256_edr_2.

In [27]:
torch.save(results_best["weights"], 'lstm_AutoEncoder_weights.pth')